# Exact suit representation tests

This notebook reproduces the representation-theory tests interactively. It derives the ambient character from fixed private hands, decomposes the 1,326-dimensional space, verifies the exact central projectors, and checks the local symmetry of a monotone river board.

In [ ]:
import sys
from pathlib import Path

import numpy as np

project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "representation.py").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from cards import make_hand
from hand_space import compatible_hand_indices
from representation import (
    CHARACTER_TABLE, CLASS_ORDER, GROUP_ORDER, PROJECTOR_NUMERATORS,
    ambient_character, irreducible_multiplicities, isotypic_dimensions,
    multiply_coefficients,
)
from river_kernel import river_kernel
from symmetry import hand_permutation, stabilizer

## 1. Derive the ambient character

For a permutation representation, the character value is the number of fixed basis coordinates. `ambient_character()` counts those coordinates directly rather than storing the expected answer.

In [ ]:
character = ambient_character()
assert character == (1326, 338, 26, 78, 0)
dict(zip(CLASS_ORDER, character))

## 2. Decompose into irreducibles

The multiplicities are exact character inner products. Multiplying each multiplicity by the corresponding irreducible dimension gives the isotypic dimensions.

In [ ]:
multiplicities = irreducible_multiplicities()
dimensions = isotypic_dimensions()

assert multiplicities == {
    "[4]": 169, "[31]": 247, "[22]": 91,
    "[211]": 78, "[1111]": 0,
}
assert sum(dimensions.values()) == 1326

[(name, CHARACTER_TABLE[name][0], multiplicities[name], dimensions[name])
 for name in CHARACTER_TABLE]

The displayed columns are `(irrep, irrep dimension, multiplicity, isotypic dimension)`. The sign representation `[1111]` is absent.

## 3. Verify the exact projector algebra

We retain the integer numerators $Q_\lambda=24P_\lambda$. Their identities therefore contain no floating-point tolerances:

$$Q_\lambda^2=24Q_\lambda,\qquad Q_\lambda Q_\mu=0,\qquad \sum_\lambda Q_\lambda=24I.$$

In [ ]:
zero = (0,) * GROUP_ORDER
for left in PROJECTOR_NUMERATORS.values():
    for right in PROJECTOR_NUMERATORS.values():
        product = multiply_coefficients(left, right)
        expected = (
            tuple(GROUP_ORDER * c for c in left.coefficients)
            if left.irrep == right.irrep else zero
        )
        assert product == expected

summed_coefficients = tuple(
    sum(projector.coefficients[i] for projector in PROJECTOR_NUMERATORS.values())
    for i in range(GROUP_ORDER)
)
assert sorted(summed_coefficients) == [0] * 23 + [24]

vector = np.arange(1326, dtype=np.int64)
resolved = sum(projector.apply(vector) for projector in PROJECTOR_NUMERATORS.values())
assert np.array_equal(resolved, 24 * vector)
print("All exact projector identities passed.")

## 4. Check the local $S_3$ symmetry

The monotone board `2c 5c 8c Jc Ac` fixes clubs and permits every permutation of the other three suits. Its stabilizer therefore has order six. The compatible hand fiber has dimension 1,081 and is preserved by this action.

In [ ]:
board = make_hand(["2c", "5c", "8c", "Jc", "Ac"])
board_stabilizer = stabilizer(board)
compatible = set(compatible_hand_indices(board))
compatibility, dominance = river_kernel(board)

assert len(board_stabilizer) == 6
assert len(compatible) == 1081

for permutation in board_stabilizer:
    indices = hand_permutation(permutation)
    coordinates = np.ix_(indices, indices)
    assert {int(indices[i]) for i in compatible} == compatible
    assert np.array_equal(compatibility[coordinates], compatibility)
    assert np.array_equal(dominance[coordinates], dominance)

print("The compatible fiber and both river operators respect the S3 stabilizer.")

## Result

The notebook has independently reproduced the automated tests: the global suit action has the claimed exact $S_4$ decomposition, its central projectors obey the integral identities, and the monotone river kernel commutes with its local board stabilizer.